# Chest X-ray visual check

**Copy this file to `$D/visual/` before running it** (`D=/lustre/isaac24/scratch/jdosch1/DiT_remote_sensing/datasets/cxr`, mode 700). An executed copy holds patient images (CheXpert, VinDr-CXR) and must never be added to git. Run in place (inside the repository) the first code cell refuses and the CheXpert and VinDr-CXR cells stay closed. This tracked file has no outputs and no execution counts; run it with the `DiTF` kernel, on a compute node, not a login node.

The cells only call `eval.cxr_visual`. Edit the paths in the first code cell, then run one section at a time. Look at the figures before any training starts.

In [ ]:
import os
import sys

# This notebook's own directory, taken before the chdir below (VS Code names the file; Jupyter starts there).
_nb = globals().get("__vsc_ipynb_file__")
NB_DIR = os.path.dirname(_nb) if _nb else globals().get("NB_DIR") or os.getcwd()
# A COPY of this notebook runs from the mode-700 scratch, so its own location is not the repository.
# Order: $CXR_REPO, then the current directory's parent, then the current directory (each must hold eval/cxr_visual.py).
_cands = [os.environ.get("CXR_REPO"), os.path.dirname(os.getcwd()), os.getcwd()]
REPO = next((d for d in _cands if d and os.path.exists(os.path.join(d, "eval", "cxr_visual.py"))), None)
if REPO is None:
    raise RuntimeError("set REPO to the repository root (or export CXR_REPO before starting Jupyter)")
D = "/lustre/isaac24/scratch/jdosch1/DiT_remote_sensing/datasets/cxr"
ROOT14 = f"{D}/trees/chestxray14_v1"  # tree roots: set to the trees you want to inspect
ROOT_CX = f"{D}/trees/chexpert_v1"
ROOT_VD = f"{D}/trees/vindr_cxr_kaggle_v2"
ARROW = f"{D}/chestxray14/arrow"
INDEX = f"{D}/chestxray14/index_v2.npz"
RAW_CX = f"{D}/chexpert/raw/chexpertchestxrays-u20210408"
RAW_VD = f"{D}/vindr_cxr/raw"
# Set to results/gates/cxr_leakage_<job id>.json (the file experiments/isaac/cxr/leakage.sbatch writes); None skips the gate cell.
GATE_JSON = None
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.chdir(REPO)
from eval import cxr_visual as V  # noqa: E402

%matplotlib inline
# Refuses when this file sits inside a git worktree or outside a mode-700 directory: an executed copy
# holds patient images. The ChestX-ray14 cells work either way; the CheXpert and VinDr-CXR cells need it.
V.notebook_guard(NB_DIR)

## ChestX-ray14 corpus (training view)

224 px images exactly as the alignment training loads them (index rows, `Cxr14Dataset`).

In [ ]:
V.corpus_grid(ARROW, INDEX, "train", n=24)

In [ ]:
V.corpus_grid(ARROW, INDEX, "val", n=24)

## ChestX-ray14 probe tree

Exported PNGs with their positive finding names, the source beside its export, the extreme aspect ratios, prevalence and sizes.

In [ ]:
V.tree_grid("chestxray14", "train", n=24, root=ROOT14)

In [ ]:
V.tree_grid("chestxray14", "test", n=24, root=ROOT14)

In [ ]:
V.raw_vs_exported("chestxray14", ARROW, n=6, root=ROOT14)

In [ ]:
V.extreme_aspect("chestxray14", ARROW, k=6, root=ROOT14)

In [ ]:
V.prevalence("chestxray14", root=ROOT14)

In [ ]:
V.size_histograms("chestxray14", root=ROOT14)

## CheXpert

Patient images: titles carry class names and the ordinal in the sample only. Keep this copy under `$D/visual/`.

In [ ]:
V.tree_grid("chexpert", "train", n=24, root=ROOT_CX)

In [ ]:
V.raw_vs_exported("chexpert", RAW_CX, n=6, root=ROOT_CX)

In [ ]:
V.extreme_aspect("chexpert", RAW_CX, k=6, root=ROOT_CX)

In [ ]:
V.prevalence("chexpert", root=ROOT_CX)

In [ ]:
V.size_histograms("chexpert", root=ROOT_CX)

## VinDr-CXR (Kaggle copy)

Patient images: ordinals and class names only. MONOCHROME1 exports beside MONOCHROME2 exports show whether the inversion is right.

In [ ]:
V.tree_grid("vindr_cxr_kaggle", "train", n=24, root=ROOT_VD)

In [ ]:
V.photometric_pairs(ROOT_VD, n=6)

In [ ]:
V.raw_vs_exported("vindr_cxr_kaggle", RAW_VD, n=6, root=ROOT_VD)

In [ ]:
V.extreme_aspect("vindr_cxr_kaggle", RAW_VD, k=6, root=ROOT_VD)

In [ ]:
V.prevalence("vindr_cxr_kaggle", root=ROOT_VD)

In [ ]:
V.size_histograms("vindr_cxr_kaggle", root=ROOT_VD)

## Leakage gate

The gate JSON as a table (counts only). Anything but PASS needs a reading before training.

In [ ]:
if GATE_JSON is None:
    print("GATE_JSON is None: set it in the first cell to show the leakage gate table")
else:
    from IPython.display import display

    display(V.gate_summary(GATE_JSON))

## Training view against probe view

Mean and max absolute pixel difference (0-255) between 1024 -> 224 (training view) and 1024 -> 448 -> 224 (probe view), on tree images that are also index rows.

In [ ]:
V.resample_gap(ARROW, INDEX, ROOT14, n=256)